##### Importing and connecting

In [1]:
import re
import sys
from pathlib import Path, PurePosixPath

import pandas as pd
from sqlalchemy import text

APPLY_TEAM_MIGRATION = True

BACKEND_DIR = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(BACKEND_DIR / "migrations"))

from common import legacy_engine, new_engine  # noqa: E402

##### Getting legacy teams

In [2]:
legacy_teams = pd.read_sql(
    text("""
        SELECT
            id,
            tname,
            tshortname,
            creation_date,
            color_1,
            color_2,
            logo,
            seasons,
            respawn_count
        FROM teams
        ORDER BY id
    """),
    legacy_engine,
)

print(f"Legacy team count: {len(legacy_teams)}")
display(legacy_teams)

Legacy team count: 188


,id,tname,tshortname,creation_date,color_1,color_2,logo,seasons,respawn_count
0,0,NCAA,NCAA,2022-11-06 23:47:47.902069,"rgba(0,0,0,0)","rgba(0,200,0,0.2)",NCAA.png,"[3, 5]",0
1,1,Miami,Miami,2022-06-06 23:44:23.380648,"rgba(244,115,33,1)","rgba(0,80,48,1)",Miami.png,"[1, 2, 3]",0
2,2,Florida Atlantic,FloridaAtlantic,2022-06-06 23:44:23.380648,"rgba(0,75,133,1)","rgba(187,47,76,1)",Florida Atlantic.png,"[1, 2, 3]",0
3,3,Florida International,FloridaInternational,2022-06-06 23:44:23.380648,"rgba(0,21,56,1)","rgba(197,150,12,1)",Florida International.png,"[1, 2, 3]",0
4,4,UCF,UCF,2022-06-06 23:44:23.380648,"rgba(186,155,55,1)","rgba(255,201,4,1)",UCF.png,"[1, 2, 3]",0
...,...,...,...,...,...,...,...,...,...
183,203,August,August,2024-12-28 07:45:45.103623,#4682B4,#B97D4B,August.png,[5],1
184,204,September,September,2024-12-28 07:45:45.103623,#6B8E23,#9471DC,September.png,[5],0
185,205,October,October,2024-12-28 07:45:45.103623,#DA70D6,#25F52F,October.png,[5],0
186,206,November,November,2024-12-28 07:45:45.103623,#40E0D0,#BF1F2F,November.png,[5],0


##### Checking turns

In [3]:
history_seasons = pd.read_sql(
    text("""
        SELECT
            tr.team AS legacy_team_id,
            array_agg(DISTINCT ti.season ORDER BY ti.season)
                FILTER (WHERE ti.season IS NOT NULL) AS history_seasons
        FROM turns AS tr
        JOIN turninfo AS ti
            ON ti.id = tr.turn_id
        GROUP BY tr.team
        ORDER BY tr.team
    """),
    legacy_engine,
)

season_audit = legacy_teams[["id", "tname", "seasons"]].merge(
    history_seasons,
    left_on="id",
    right_on="legacy_team_id",
    how="left",
)

display(season_audit)

,id,tname,seasons,legacy_team_id,history_seasons
0,0,NCAA,"[3, 5]",NaN,NaN
1,1,Miami,"[1, 2, 3]",1.0,"[1, 2, 3]"
2,2,Florida Atlantic,"[1, 2, 3]",2.0,[1]
3,3,Florida International,"[1, 2, 3]",NaN,NaN
4,4,UCF,"[1, 2, 3]",4.0,"[1, 2, 3]"
...,...,...,...,...,...
183,203,August,[5],203.0,[5]
184,204,September,[5],204.0,[5]
185,205,October,[5],205.0,[5]
186,206,November,[5],206.0,[5]


##### Color

In [4]:
RGBA_PATTERN = re.compile(
    r"rgba?\(\s*(\d{1,3})\s*,\s*(\d{1,3})\s*,\s*"
    r"(\d{1,3})(?:\s*,\s*(0|1|0?\.\d+))?\s*\)",
    re.IGNORECASE,
)
HEX_PATTERN = re.compile(r"^#[0-9A-Fa-f]{6}$")


def normalize_color(value):
    """Normalize RGB/RGBA or hex color values to #RRGGBB."""
    if value is None or pd.isna(value):
        return None

    value = str(value).strip()

    if not value:
        return None

    if HEX_PATTERN.fullmatch(value):
        return value.upper()

    match = RGBA_PATTERN.fullmatch(value)

    if match:
        r, g, b = (int(channel) for channel in match.groups()[:3])

        if any(channel > 255 for channel in (r, g, b)):
            raise ValueError(f"Invalid RGB color: {value}")

        return f"#{r:02X}{g:02X}{b:02X}"

    raise ValueError(f"Unrecognized color format: {value!r}")


teams_review = legacy_teams.copy()

teams_review["primary_color"] = teams_review["color_1"].map(normalize_color)
teams_review["secondary_color"] = teams_review["color_2"].map(normalize_color)

#### Extra teams

In [5]:

legacy_teams = teams_review.copy()

legacy_teams["tname"] = legacy_teams["tname"].astype("string").str.strip()

# Only Undecimber maps to Chaos.
legacy_teams.loc[
    legacy_teams["tname"].str.casefold() == "undecimber",
    "tname",
] = "Chaos"

# Use the Hawaiian ʻokina (U+02BB) consistently.
legacy_teams.loc[
    legacy_teams["tname"].str.casefold().isin(["hawai'i", "hawaiʻi"]),
    "tname",
] = "Hawaiʻi"


def normalized_name(value):
    if value is None or pd.isna(value):
        return ""
    return str(value).strip().casefold()


LOGO_FILENAME_OVERRIDES = {
    "baldeagle": "Bald Eagle",
    "bluejay": "Blue Jay",
    "jacksonvillestate": "Jacksonville State",
    "jamesmadison": "James Madison",
    "kennesawstate": "Kennesaw State",
    "missouristate": "Missouri State",
    "mountainlion": "Mountain Lion",
    "northdakotastate": "North Dakota State",
    "prairiedog": "Prairie Dog",
    "sacramentostate": "Sacramento State",
    "samh ouston".replace(" ", ""): "Sam Houston",
    "hawai'i": "Hawaiʻi",
    "hawaiʻi": "Hawaiʻi",
}


def normalize_logo_svg(value, fallback_name):
    """
    Normalize logo extension and correct known logo filenames.
    Preserve spaces in filenames when required by frontend assets.
    """
    if value is None or pd.isna(value) or not str(value).strip():
        logo = str(fallback_name).strip()
        suffix = ".svg"
    else:
        path = PurePosixPath(str(value).strip().replace("\\", "/"))
        logo = path.stem
        suffix = ".svg"

    # Match filenames regardless of existing spaces or capitalization.
    compact_stem = re.sub(r"\s+", "", logo).casefold()

    if compact_stem in LOGO_FILENAME_OVERRIDES:
        logo = LOGO_FILENAME_OVERRIDES[compact_stem]

    # Preserve the original directory, if present.
    if value is not None and not pd.isna(value) and str(value).strip():
        return str(path.with_name(f"{logo}{suffix}"))

    return f"{logo}{suffix}"



NEW_FBS_TEAM_COLORS = {
    "Delaware": {
        "primary_color": "#00539F",
        "secondary_color": "#FFDd31",
    },
    "Jacksonville State": {
        "primary_color": "#CC0000",
        "secondary_color": "#FFFFFF",
    },
    "James Madison": {
        "primary_color": "#450084",
        "secondary_color": "#CBB677",
    },
    "Kennesaw State": {
        "primary_color": "#fdbb30",
        "secondary_color": "#0b1315",
    },
    "Missouri State": {
        "primary_color": "#5E0009",
        "secondary_color": "#FFFFFF",
    },
    "North Dakota State": {
        "primary_color": "#0a5640",
        "secondary_color": "#ffc72a",
    },
    "Sacramento State": {
        "primary_color": "#043927",
        "secondary_color": "#c4b581",
    },
    "Sam Houston": {
        "primary_color": "#fe5100",
        "secondary_color": "#FFFFFF",
    },
}

NEW_FBS_TEAMS = list(NEW_FBS_TEAM_COLORS.keys())

assert len(NEW_FBS_TEAMS) == 8
assert all(
    HEX_PATTERN.fullmatch(colors["primary_color"])
    and HEX_PATTERN.fullmatch(colors["secondary_color"])
    for colors in NEW_FBS_TEAM_COLORS.values()
)


def normalize_seasons(value):
    """Convert a legacy seasons array into a sorted list of integers."""
    if value is None:
        return []

    if not isinstance(value, (list, tuple)):
        if pd.isna(value):
            return []

    return sorted({int(season) for season in value if season is not None and not pd.isna(season)})


special_names = {"chaos", "ncaa"}

original_fbs = legacy_teams.loc[
    legacy_teams["seasons"].map(lambda seasons: {1, 2, 3}.issubset(set(normalize_seasons(seasons))))
    & ~legacy_teams["tname"].map(normalized_name).isin(special_names)
].copy()

if len(original_fbs) != 130:
    raise ValueError(
        f"Expected 130 original FBS teams, found {len(original_fbs)}. "
        "Review the legacy seasons and team exclusions before continuing."
    )

duplicate_names = original_fbs["tname"].map(normalized_name)
duplicate_names = duplicate_names[duplicate_names.duplicated(keep=False)]

if not duplicate_names.empty:
    raise ValueError(
        f"Duplicate original FBS names found: {sorted(duplicate_names.unique().tolist())}"
    )

original_names = set(original_fbs["tname"].map(normalized_name))

overlapping_new_teams = [name for name in NEW_FBS_TEAMS if normalized_name(name) in original_names]

if overlapping_new_teams:
    raise ValueError(
        f"These new FBS teams already appear in the original roster: {overlapping_new_teams}"
    )

original_rows = pd.DataFrame(
    {
        "legacy_team_id": original_fbs["id"].astype("int64").to_numpy(),
        "name": original_fbs["tname"].to_numpy(),
        "primary_color": original_fbs["primary_color"].to_numpy(),
        "secondary_color": original_fbs["secondary_color"].to_numpy(),
        "logo": original_fbs.apply(
            lambda row: normalize_logo_svg(row["logo"], row["tname"]),
            axis=1,
        ).to_numpy(),
        "seasons": [[1, 2, 3] for _ in range(len(original_fbs))],
    }
)

new_fbs_rows = pd.DataFrame(
    [
        {
            "legacy_team_id": pd.NA,
            "name": name,
            "primary_color": colors["primary_color"],
            "secondary_color": colors["secondary_color"],
            "logo": normalize_logo_svg(None, name),
            "seasons": [],
        }
        for name, colors in NEW_FBS_TEAM_COLORS.items()
    ]
)


chaos_source = legacy_teams.loc[legacy_teams["tname"].map(normalized_name) == "chaos"].copy()

if len(chaos_source) != 1:
    raise ValueError(
        f"Expected exactly one Undecimber/Chaos source row, found {len(chaos_source)}."
    )

chaos_row = chaos_source.iloc[0]

chaos_df = pd.DataFrame(
    [
        {
            "legacy_team_id": int(chaos_row["id"]),
            "name": "Chaos",
            "primary_color": chaos_row["primary_color"],
            "secondary_color": chaos_row["secondary_color"],
            "logo": normalize_logo_svg(chaos_row["logo"], "Chaos"),
            "seasons": [1, 2, 3, 4, 5],
        }
    ]
)


fbs_df = pd.concat(
    [original_rows, new_fbs_rows],
    ignore_index=True,
)

fbs_df = fbs_df.sort_values(
    "name",
    key=lambda column: column.str.casefold(),
).reset_index(drop=True)

fbs_df["display_order"] = range(1, len(fbs_df) + 1)
chaos_df["display_order"] = 0

roster_139 = pd.concat(
    [chaos_df, fbs_df],
    ignore_index=True,
)

roster_139["display_order"] = roster_139["display_order"].astype("int64")

# Every logo path must end in .svg.
# Every logo path must end in .svg, case-insensitively.
invalid_logos = roster_139.loc[
    ~roster_139["logo"].astype("string").str.lower().str.endswith(".svg", na=False),
    ["name", "logo"],
]

if not invalid_logos.empty:
    raise ValueError(f"Some logo paths do not end in .svg:\n{invalid_logos.to_string(index=False)}")
display(
    roster_139[
        [
            "display_order",
            "legacy_team_id",
            "name",
            "primary_color",
            "secondary_color",
            "logo",
            "seasons",
        ]
    ]
)

,display_order,legacy_team_id,name,primary_color,secondary_color,logo,seasons
0,0,131,Chaos,#000000,#0C7F0B,Chaos.svg,"[1, 2, 3, 4, 5]"
1,1,106,Air Force,#004A7B,#FFFFFF,Air Force.svg,"[1, 2, 3]"
2,2,30,Akron,#00285E,#84754E,Akron.svg,"[1, 2, 3]"
3,3,58,Alabama,#9E1B32,#828A8F,Alabama.svg,"[1, 2, 3]"
4,4,39,Appalachian State,#000000,#FFCD00,Appalachian State.svg,"[1, 2, 3]"
...,...,...,...,...,...,...,...
134,134,33,West Virginia,#002855,#EAAA00,West Virginia.svg,"[1, 2, 3]"
135,135,85,Western Kentucky,#F32026,#B3B5B8,Western Kentucky.svg,"[1, 2, 3]"
136,136,95,Western Michigan,#6C4023,#B5A167,Western Michigan.svg,"[1, 2, 3]"
137,137,100,Wisconsin,#C5050C,#FFFFFF,Wisconsin.svg,"[1, 2, 3]"


##### Writing to SQL

In [6]:
if not APPLY_TEAM_MIGRATION:
    raise RuntimeError(
        "Import disabled. Review preview, then set APPLY_TEAM_MIGRATION = True in the preview cell."
    )

if len(roster_139) != 139:
    raise ValueError("Expected 139 teams.")

if roster_139["name"].map(normalized_name).duplicated().any():
    raise ValueError("Duplicate team names found.")

chaos = roster_139.loc[roster_139["name"].map(normalized_name) == "chaos"]

if len(chaos) != 1:
    raise ValueError("Expected exactly one Chaos row.")

chaos = chaos.iloc[0]

# Never overwrite an existing target roster implicitly.
with new_engine.begin() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM public.teams")).scalar_one()

    if count != 0:
        raise RuntimeError(
            f"Target contains {count} teams. Reconcile existing "
            "records before importing; no changes were committed."
        )

    # Insert Chaos with explicit target ID 0.
    # Handle GENERATED ALWAYS identity columns correctly.
    identity_generation = conn.execute(
        text("""
            SELECT identity_generation
            FROM information_schema.columns
            WHERE table_schema = 'public'
              AND table_name = 'teams'
              AND column_name = 'id'
        """)
    ).scalar_one_or_none()

    chaos_insert = """
        INSERT INTO public.teams
            (id, legacy_team_id, name, primary_color,
             secondary_color, logo, seasons)
        {override}
        VALUES
            (:id, :legacy_team_id, :name, :primary_color,
             :secondary_color, :logo, :seasons)
    """.format(override=("OVERRIDING SYSTEM VALUE" if identity_generation == "ALWAYS" else ""))

    conn.execute(
        text(chaos_insert),
        {
            "id": 0,
            "legacy_team_id": int(chaos["legacy_team_id"]),
            "name": "Chaos",
            "primary_color": chaos["primary_color"],
            "secondary_color": chaos["secondary_color"],
            "logo": chaos["logo"],
            "seasons": list(chaos["seasons"]),
        },
    )

    # Insert remaining teams without specifying id.
    other_teams = roster_139.loc[roster_139["name"].map(normalized_name) != "chaos"]

    insert_sql = text("""
        INSERT INTO public.teams
            (legacy_team_id, name, primary_color,
             secondary_color, logo, seasons)
        VALUES
            (:legacy_team_id, :name, :primary_color,
             :secondary_color, :logo, :seasons)
    """)

    records = []
    for row in other_teams.to_dict("records"):
        legacy_id = row["legacy_team_id"]
        records.append(
            {
                "legacy_team_id": (None if pd.isna(legacy_id) else int(legacy_id)),
                "name": row["name"],
                "primary_color": row["primary_color"],
                "secondary_color": row["secondary_color"],
                "logo": row["logo"],
                "seasons": list(row["seasons"]),
            }
        )

    conn.execute(insert_sql, records)

    # Reset sequence so the next generated ID is above current maximum.
    sequence_name = conn.execute(
        text("SELECT pg_get_serial_sequence('public.teams', 'id')")
    ).scalar_one()

    if sequence_name is None:
        raise RuntimeError("teams.id has no associated sequence; inspect schema.")

    conn.execute(
        text("""
            SELECT setval(
                CAST(:sequence_name AS regclass),
                GREATEST((SELECT MAX(id) FROM public.teams), 1),
                (SELECT MAX(id) FROM public.teams) >= 1
            )
        """),
        {"sequence_name": sequence_name},
    )

    # Transaction-level verification before commit.
    final_count = conn.execute(text("SELECT COUNT(*) FROM public.teams")).scalar_one()

    chaos_id = conn.execute(text("SELECT id FROM public.teams WHERE name = 'Chaos'")).scalar_one()

    if final_count != 139:
        raise RuntimeError(f"Expected 139 rows, got {final_count}.")

    if chaos_id != 0:
        raise RuntimeError(f"Expected Chaos ID 0, got {chaos_id}.")

print("Imported 139 teams successfully.")

Imported 139 teams successfully.


In [7]:
# Calendar ordering for Season 5
month_order = {
    month: i
    for i, month in enumerate(
        [
            "January",
            "February",
            "March",
            "April",
            "May",
            "June",
            "July",
            "August",
            "September",
            "October",
            "November",
            "December",
        ]
    )
}

# Find all legacy teams assigned to Season 4 or 5.
# NCAA is the only legacy team excluded; existing imports are skipped below.
additions = legacy_teams.copy()
additions["seasons_list"] = additions["seasons"].map(normalize_seasons)
additions["name"] = additions["tname"].astype(str).str.strip()

additions = additions.loc[
    additions["seasons_list"].map(lambda s: 4 in s or 5 in s)
    & additions["name"].map(normalized_name).ne("ncaa")
].copy()

# Apply requested naming and logo corrections.
blue_jay = additions["name"].map(normalized_name).eq("blue jay")

# Apply consistent team-name and logo corrections.
hawaii_mask = additions["name"].str.casefold().isin(["hawai'i", "hawaiʻi"])
additions.loc[hawaii_mask, "name"] = "Hawaiʻi"

additions["logo"] = additions.apply(
    lambda row: normalize_logo_svg(row["logo"], row["name"]),
    axis=1,
)


# Order Season 4 alphabetically, then Season 5 January through December.
additions["sort_season"] = additions["seasons_list"].map(lambda s: 4 if 4 in s else 5)
additions["month_order"] = additions["name"].map(lambda name: month_order.get(name, 99))
additions["sort_name"] = additions["name"].str.casefold()

additions = additions.sort_values(["sort_season", "month_order", "sort_name"])

# Only skip teams already imported, so running this cell again is safe.
with new_engine.begin() as conn:
    existing = (
        conn.execute(
            text("""
            SELECT legacy_team_id, name
            FROM teams
        """)
        )
        .mappings()
        .all()
    )

    existing_legacy_ids = {
        row["legacy_team_id"] for row in existing if row["legacy_team_id"] is not None
    }
    existing_names = {normalized_name(row["name"]) for row in existing}

    rows_to_insert = []

    for _, row in additions.iterrows():
        legacy_id = int(row["id"])
        name = row["name"]

        if legacy_id in existing_legacy_ids:
            continue
        if normalized_name(name) in existing_names:
            continue

        rows_to_insert.append(
            {
                "legacy_team_id": legacy_id,
                "name": name,
                "primary_color": normalize_color(row["color_1"]),
                "secondary_color": normalize_color(row["color_2"]),
                "logo": row["logo"],
                "seasons": row["seasons_list"],
            }
        )

        # Prevent duplicates within this batch.
        existing_legacy_ids.add(legacy_id)
        existing_names.add(normalized_name(name))

    if rows_to_insert:
        conn.execute(
            text("""
                INSERT INTO teams (
                    legacy_team_id,
                    name,
                    primary_color,
                    secondary_color,
                    logo,
                    seasons
                )
                VALUES (
                    :legacy_team_id,
                    :name,
                    :primary_color,
                    :secondary_color,
                    :logo,
                    :seasons
                )
            """),
            rows_to_insert,
        )

print(f"Added {len(rows_to_insert)} Season 4/5 teams.")

Added 56 Season 4/5 teams.
